# Union vs Union all

### Potential interview question
1. Differrence between union and union all ?
2. What will happen if i change the no of the column while unioning the data ?
3. What if column name is different ?
4. Waht is unionByName ?

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("unionVsUnionAllTestApp")\
                    .master("local[*]")\
                    .getOrCreate()
spark

In [2]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(18 ,'Sam',55000,   17),
(18 ,'Sam',65000,   17)
]
schema = ["id", "name", "sal", "mngr_id"]

managerDf = spark.createDataFrame(data=data, schema=schema)

In [3]:
managerDf.show()
managerDf.count()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| Vikas|75000|     16|
| 12| Nisha|40000|     18|
| 13| Nidhi|60000|     17|
| 14| Priya|80000|     18|
| 15| Mohit|45000|     18|
| 16|Rajesh|90000|     10|
| 17| Raman|55000|     16|
| 18|   Sam|65000|     17|
| 18|   Sam|55000|     17|
| 18|   Sam|65000|     17|
+---+------+-----+-------+



11

In [4]:
data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17),
(18 ,'Sam',65000,   17)]
schema1 = ["id", "name", "sal", "mngr_id"]

managerDf1 = spark.createDataFrame(data=data1, schema=schema1)

In [5]:
managerDf1.count()

3

## union
- same as unionAll and return consolidated records from both `dataframe`
- `Removes duplicates` (similar to SQL UNION) if you are `using the spark SQL`
- Requires both DataFrames to have the `same schema`

### important
When you using dataframe both `union` and `unionAll` work same.

## unionAll
- Includes all rows, including duplicates
- Was available in older versions (before Spark 2.0)
- Replaced by union() — you now do

```python
df1.union(df2)  # includes duplicates
df1.union(df2).distinct()  # removes duplicates
```

In [6]:
unionDf = managerDf.union(managerDf1)
unionAllDf = managerDf.unionAll(managerDf1)
print(f"count of union df : {unionDf.count()}") # duplicate will not remove in DF
print(f"count of unionAll df : {unionAllDf.count()}")

count of union df : 14
count of unionAll df : 14


##### LETS TEST  UNION IN SPARK SQL
**`Removes duplicates` (similar to SQL UNION) if you are `using the spark SQL`**

In [7]:
managerDf.createOrReplaceTempView("managerTbl")
managerDf1.createOrReplaceTempView("managerTbl1")
unionSql = spark.sql("select * from managerTbl union select * from managerTbl1")
unionAllSql = spark.sql("select * from managerTbl union all select * from managerTbl1")
print(f"union count in sparkSQL : {unionSql.count()}") # duplicates removed in SparkSQL not DF
print(f"union all count in sparkSQL : {unionAllSql.count()}") # include duplicates


union count in sparkSQL : 12
union all count in sparkSQL : 14


### 3. What if column name is different ?
🧠 Tip:
Always make sure:
- Same number of columns
- Same column names
- Same data types (cast if needed)

In [39]:
managerDf1.show()

+---+-----+-----+-------+
| id| name|  sal|mngr_id|
+---+-----+-----+-------+
| 19|Sohan|50000|     18|
| 20| Sima|75000|     17|
| 18|  Sam|65000|     17|
+---+-----+-----+-------+



In [ ]:
wrong_column_data=[(19 ,5000, 18,'Sohang')]

wrong_schema = ["id", "sal", "mngr_id", "name"]

wrongManagerDf = spark.createDataFrame(data=wrong_column_data, schema=wrong_schema)

In [37]:
# managerDf1.union(wrongManagerDf).show() # it will failed as conlumn order is not matching

### ⚠️`unionByName()` in PySpark

While `unionByName()` simplifies combining DataFrames by column names, several issues can occur:

1. **Mismatched Column Names**:

   * Columns must have **exact same names** (case-sensitive).
   * Solution: Rename columns using `withColumnRenamed()`.

2. **Different Data Types**:

   * Columns with same names but different types cause errors.
   * Solution: Cast to matching types using `.cast()`.

3. **Missing Columns**:

   * If one DataFrame has extra columns, it fails unless:
   * Solution: Use `allowMissingColumns=True` (PySpark 3.1+).

4. **Nested or Complex Types**:

   * `struct`, `array`, or `map` columns must have **identical structure**.
   * Solution: Align schema or flatten before union.

5. **Version Compatibility**:

   * `allowMissingColumns` is not available in PySpark < 3.1.

**Tip:** Always check schema using `printSchema()` before using `unionByName()`.


In [40]:
managerDf1.unionByName(wrongManagerDf).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 19| Sohan|50000|     18|
| 20|  Sima|75000|     17|
| 18|   Sam|65000|     17|
| 19|Sohang| 5000|     18|
+---+------+-----+-------+

